# Make Models: Band-Only Baseline

B Steele, Colorado State University, ROSSyndicate — last updated Oct 8, 2026

**Pipeline step 04b · Python · run from `regional_clarity/`**

- **Purpose:** Train a band-only counterpart to the production ensemble with identical logic, so that the contribution of the static site characteristics and antecedent weather can be measured directly.
- **Inputs:** `aquamatch_files/base_with_splits.parquet`; `xg_models/v3_production/seed{601–610}/holdout_predictions.parquet` (for the comparison).
- **Outputs:** `xg_models/v3_band_only/seed{601–610}/`: `backward_elim_summary.json`, `final_tune_xgboost.json`, `xgboost_fold{1–4}.json`, `final_eval_summary.json`, `holdout_predictions.parquet`; for the full-data refit, `xgboost_full.json` and `holdout_predictions_full.parquet`.
- **Previous:** [04 · Make models](04_make_models.ipynb)
- **Next:** [05b · Evaluate ensemble (band-only)](05b_evaluate_ensemble_band_only.ipynb)

This notebook repeats `04_make_models.ipynb` step for step: same input data, same fixed holdout, same 10 seeds and the same per-seed 4-fold HUC8 arrangements, correlation pruning, gap-aware tuning, shadow-decoy backward elimination, and unweighted final training. The only difference is the starting candidate pool, which is restricted to information derived from the Landsat scene itself:

- `atm_corr_LaSRC` (atmospheric-correction processing stream);
- the seven harmonized surface-reflectance/temperature bands (`*_corr7`);
- the nine spectral ratios and indices from `features.add_spectral_indices()`.

Site characteristics (elevation, LakeCat catchment land cover), `shore_flag`, and all antecedent gridMET weather summaries are excluded. The candidate pool is an explicit whitelist rather than "everything except site and weather columns," so that no non-spectral column can enter by default.

Because every seed's CV arrangement and the holdout are identical between the two runs, each band-only seed is directly paired with its production counterpart. Any difference in performance reflects the feature pool, not the data split.

**Final models: fold ensemble and full-data refit.** There are two standard ways to turn a cross-validated model selection into final models. The fold ensemble used here keeps the four models each seed trains during CV and averages them, so every member is early-stopped against HUC8s it never trained on. The alternative, standard in species-distribution modeling (ENMeval for Maxent, for example), uses CV only to choose settings and then refits one model on all of the training data with them. Each seed builds both from the same features and hyperparameters: alongside its four fold models it fits one model on the full CV pool. That model's boosting-round count is the mean of the four fold models' early-stopped round counts, because no held-out data remain to early-stop against. The 10 full-data refits form a 10-model ensemble, which `05b_evaluate_ensemble_band_only.ipynb` compares with the 40-model fold ensemble and `06b_test_set_sensitivity_band_only.ipynb` evaluates across holdout partitions.

## Setup

Same library and run parameters as `04_make_models.ipynb`; only the output directory and candidate pool differ.

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("python").resolve()))
import backward_elim as be
import features
import metrics as M
import model_xgb
import spatial_cv
from tuning import gap_aware_tune

be.register_model("xgboost", model_xgb)

AQUAMATCH_DIR = Path("aquamatch_files")
MODELS_DIR = Path("xg_models") / "v3_band_only"
PRODUCTION_DIR = Path("xg_models") / "v3_production"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "harmonized_value"
MODEL_NAME = "xgboost"
MOD = model_xgb
SEEDS = list(range(601, 611))
N_TRIALS_INITIAL = 25
N_TRIALS_FINAL = 40
ELIM_TOL = 0.005
ELIM_SEED = 47
TUNE_SEED = 47

# band-only candidate pool: scene-derived information only (no site characteristics,
# shore_flag, or antecedent weather)
INDEX_COLS = ["BR", "BG", "NR", "GR", "fai", "NDVI", "NDSSI", "NDWI", "MNDWI"]
BAND_ONLY_CANDIDATES = ["atm_corr_LaSRC"] + features.BASE_BANDS + INDEX_COLS


def log(seed, msg):
    print(f"[{time.strftime('%H:%M:%S')}][seed{seed}] {msg}", flush=True)

## Load data and build the same 10 CV-fold arrangements

Identical to `04`: `spatial_cv.build_splits()` is deterministic given the seeds, so the holdout and each seed's folds match production exactly. `coarsen_site_features()` is still applied so the dataframe matches `04`'s, though none of its columns are candidates here.

In [2]:
df = pd.read_parquet(AQUAMATCH_DIR / "base_with_splits.parquet")
df = features.add_spectral_indices(df)
df = features.coarsen_site_features(df)
df = spatial_cv.build_splits(df, ensemble_seeds=SEEDS)
print(f"{len(df):,} rows, {df.shape[1]} columns; holdout n={df['is_holdout'].sum():,}")
assert all(c in df.columns for c in BAND_ONLY_CANDIDATES)

11,143 rows, 72 columns; holdout n=2,206


## Per-seed pipeline (band-only candidates)

`run_seed_pipeline()` is the function from `04`, with only the candidate-pool construction changed. Each step (pruning, initial tuning, backward elimination, final tuning, training, evaluation, and the full-data refit) is otherwise unchanged.

In [3]:
def run_seed_pipeline(seed, df):
    out_dir = MODELS_DIR / f"seed{seed}"
    out_dir.mkdir(parents=True, exist_ok=True)
    t_start = time.time()

    folds = spatial_cv.build_folds(df, seed)
    test = df[df["is_holdout"]].reset_index(drop=True)
    log(seed, f"folds: {[(f.part, len(f.train), len(f.val)) for f in folds]}, holdout n={len(test)}")

    # --- candidate pool + correlation pruning (this seed's own train_val pool) ---
    candidates = list(BAND_ONLY_CANDIDATES)

    train_val = pd.concat([f.train for f in folds] + [folds[0].val], ignore_index=True).drop_duplicates(
        subset=["siteSR_id", "date"])
    pruned = features.correlation_prune(train_val, candidates, target=TARGET)
    log(seed, f"candidate pool: {len(candidates)} -> {len(pruned)} after correlation pruning")

    # tuning seed is derived per-ensemble-seed (rather than a single fixed value shared
    # by every seed) so each seed's random search explores a genuinely different draw of
    # hyperparameter combinations - previously every seed reused seed=47 verbatim, so all
    # 10 seeds evaluated the identical set of combinations against their own data instead
    # of diversifying search coverage across the ensemble the way the CV folds and feature
    # selection already do
    seed_tune_seed = TUNE_SEED + seed

    # --- stage 1: initial gap-aware tuning (fixed params for elimination) ---
    log(seed, f"=== initial gap-aware tuning on {len(pruned)}-feature pool ===")
    init_result = gap_aware_tune(MOD, folds, pruned, TARGET, n_trials=N_TRIALS_INITIAL, seed=seed_tune_seed)
    fixed_params = init_result["best_params"]
    log(seed, f"val_rmse={init_result['best_score']:.4f} gap={init_result['best_gap']:.4f}")

    # --- stage 2: backward elimination (single algorithm, no joint step) ---
    log(seed, "=== backward elimination ===")
    elim_result = be.backward_eliminate(MODEL_NAME, folds, TARGET, fixed_params, pruned,
                                        weight_fn=None, tol=ELIM_TOL, seed=ELIM_SEED,
                                        log=lambda m: log(seed, m))
    final_feats = elim_result["final_features"]
    log(seed, f"final ({len(final_feats)}): {final_feats}")

    with open(out_dir / "backward_elim_summary.json", "w") as fh:
        json.dump(dict(starting_candidates=candidates, correlation_pruned=pruned,
                       final_features=final_feats, path=elim_result["path"]), fh, indent=2, default=str)

    # --- stage 3: final gap-aware tuning on the final feature set ---
    log(seed, f"=== final gap-aware tuning on {len(final_feats)}-feature set ===")
    tune_result = gap_aware_tune(MOD, folds, final_feats, TARGET, n_trials=N_TRIALS_FINAL, seed=seed_tune_seed)
    log(seed, f"chosen: val_rmse={tune_result['best_score']:.4f} gap={tune_result['best_gap']:.4f}")
    with open(out_dir / f"final_tune_{MODEL_NAME}.json", "w") as fh:
        json.dump({"model": MODEL_NAME, "features": final_feats, **tune_result}, fh, indent=2, default=str)

    # --- stage 4: final unweighted training + evaluation ---
    params = tune_result["best_params"]
    fold_models = MOD.train_fold_models(folds, final_feats, TARGET, params)
    for i, fm in enumerate(fold_models):
        fm.save_model(str(out_dir / f"{MODEL_NAME}_fold{i+1}.json"))

    cv_preds, cv_ys = [], []
    for f, model in zip(folds, fold_models):
        cv_preds.append(MOD.predict_ensemble([model], f.val[final_feats]))
        cv_ys.append(f.val[TARGET].values)
    cv_metrics = M.all_metrics(np.concatenate(cv_ys), np.concatenate(cv_preds))

    test_pred = MOD.predict_ensemble(fold_models, test[final_feats])
    test_metrics = M.all_metrics(test[TARGET].values, test_pred)
    log(seed, f"FINAL: cv_rmse={cv_metrics['rmse']:.4f} holdout_rmse={test_metrics['rmse']:.4f}")

    summary = {MODEL_NAME: dict(features=final_feats, params=params, cv_metrics=cv_metrics, test_metrics=test_metrics)}
    with open(out_dir / "final_eval_summary.json", "w") as fh:
        json.dump(summary, fh, indent=2, default=str)

    r = test[["siteSR_id", "date", "HUC4", "HUC8"]].copy()
    r["seed"] = seed
    r["model"] = MODEL_NAME
    r["y"] = test[TARGET].values
    r["pred"] = test_pred
    r.to_parquet(out_dir / "holdout_predictions.parquet")

    # --- stage 5: full-data refit (ENMeval-style alternative to the fold ensemble) ---
    # same CV-selected features and hyperparameters, one model fit on the whole CV pool.
    # with no held-out data left to early-stop against, the round count is the mean of
    # the four fold models' early-stopped round counts
    num_rounds = MOD.cv_num_rounds(fold_models)
    cv_pool = df[~df["is_holdout"]].reset_index(drop=True)
    full_model = MOD.train_full_model(cv_pool, final_feats, TARGET, params, num_rounds)
    full_model.save_model(str(out_dir / f"{MODEL_NAME}_full.json"))

    full_pred = MOD.predict_ensemble([full_model], test[final_feats])
    full_metrics = M.all_metrics(test[TARGET].values, full_pred)
    log(seed, f"FULL REFIT ({num_rounds} rounds): holdout_rmse={full_metrics['rmse']:.4f}")

    summary[MODEL_NAME]["full_refit"] = dict(num_rounds=num_rounds,
                                             fold_rounds=[m.num_boosted_rounds() for m in fold_models],
                                             test_metrics=full_metrics)
    with open(out_dir / "final_eval_summary.json", "w") as fh:
        json.dump(summary, fh, indent=2, default=str)
    r.assign(model=f"{MODEL_NAME}_full", pred=full_pred).to_parquet(out_dir / "holdout_predictions_full.parquet")

    log(seed, f"SEED {seed} DONE in {time.time()-t_start:.0f}s")
    return summary

Run the pipeline for all 10 seeds.

In [4]:
all_summaries = {}
for seed in SEEDS:
    all_summaries[seed] = run_seed_pipeline(seed, df)

[17:58:51][seed601] folds: [(1, 6764, 2173), (2, 6551, 2386), (3, 6822, 2115), (4, 6674, 2263)], holdout n=2206


[17:58:51][seed601] candidate pool: 17 -> 13 after correlation pruning


[17:58:51][seed601] === initial gap-aware tuning on 13-feature pool ===


[17:59:54][seed601] val_rmse=1.5759 gap=0.1279


[17:59:54][seed601] === backward elimination ===


...

[18:36:34][seed610] === final gap-aware tuning on 13-feature set ===


[18:38:21][seed610] chosen: val_rmse=1.5720 gap=0.0937


[18:38:23][seed610] FINAL: cv_rmse=1.5830 holdout_rmse=1.3540


[18:38:23][seed610] FULL REFIT (316 rounds): holdout_rmse=1.3988


[18:38:23][seed610] SEED 610 DONE in 218s


## Per-seed summary

In [5]:
rows = []
for seed, summary in all_summaries.items():
    s = summary[MODEL_NAME]
    rows.append(dict(seed=seed, n_features=len(s["features"]), cv_rmse=s["cv_metrics"]["rmse"],
                     holdout_rmse=s["test_metrics"]["rmse"], holdout_bias=s["test_metrics"]["bias"],
                     full_rounds=s["full_refit"]["num_rounds"],
                     holdout_rmse_full=s["full_refit"]["test_metrics"]["rmse"],
                     holdout_bias_full=s["full_refit"]["test_metrics"]["bias"]))
pd.DataFrame(rows).sort_values("seed")

,seed,n_features,cv_rmse,holdout_rmse,holdout_bias,full_rounds,holdout_rmse_full,holdout_bias_full
0,601,12,1.583240,1.368597,0.084870,1536,1.396538,0.117044
1,602,13,1.587185,1.373792,0.112172,1344,1.397908,0.132144
2,603,13,1.585084,1.382970,0.114506,1340,1.393833,0.127057
3,604,12,1.558675,1.393852,0.132486,1872,1.400649,0.133435
4,605,13,1.575118,1.379942,0.125899,373,1.404277,0.128479
5,606,13,1.591548,1.366263,0.107057,2249,1.398631,0.133932
6,607,13,1.567268,1.356524,0.102513,440,1.401945,0.127405
7,608,13,1.580567,1.370145,0.116100,1474,1.395579,0.129596
8,609,13,1.569334,1.377083,0.115496,662,1.412003,0.133512
9,610,13,1.582973,1.353955,0.075659,316,1.398827,0.130831


## Comparison against production

Each band-only seed is paired with the production seed that has the same CV arrangement, so the per-seed differences isolate the effect of the feature pool. Both ensembles are scored on the same holdout rows. Negative ΔRMSE means production (bands + site + weather) is better.

In [6]:
def load_preds(models_dir):
    return pd.concat([pd.read_parquet(models_dir / f"seed{s}" / "holdout_predictions.parquet") for s in SEEDS],
                     ignore_index=True)

preds = {"band_only": load_preds(MODELS_DIR), "production": load_preds(PRODUCTION_DIR)}

# per-seed, paired
rows = []
for seed in SEEDS:
    r = dict(seed=seed)
    for arm, p in preds.items():
        s = p[p["seed"] == seed]
        m = M.all_metrics(s["y"], s["pred"])
        r[f"{arm}_rmse"], r[f"{arm}_bias"] = m["rmse"], m["bias"]
    rows.append(r)
per_seed = pd.DataFrame(rows)
per_seed["delta_rmse"] = per_seed["production_rmse"] - per_seed["band_only_rmse"]
per_seed["delta_rmse_pct"] = 100 * per_seed["delta_rmse"] / per_seed["band_only_rmse"]
print(f"production better on {(per_seed['delta_rmse'] < 0).sum()}/{len(SEEDS)} seeds; "
      f"mean ΔRMSE {per_seed['delta_rmse'].mean():+.3f} m ({per_seed['delta_rmse_pct'].mean():+.1f}%), "
      f"sd {per_seed['delta_rmse'].std():.3f} m")
per_seed.round(3)

production better on 10/10 seeds; mean ΔRMSE -0.102 m (-7.4%), sd 0.013 m


,seed,band_only_rmse,band_only_bias,production_rmse,production_bias,delta_rmse,delta_rmse_pct
0,601,1.369,0.085,1.271,0.182,-0.098,-7.147
1,602,1.374,0.112,1.280,0.201,-0.094,-6.841
2,603,1.383,0.115,1.272,0.161,-0.111,-8.007
3,604,1.394,0.132,1.275,0.183,-0.119,-8.520
4,605,1.380,0.126,1.267,0.181,-0.113,-8.198
5,606,1.366,0.107,1.274,0.151,-0.093,-6.780
6,607,1.357,0.103,1.276,0.173,-0.081,-5.957
7,608,1.370,0.116,1.258,0.192,-0.113,-8.213
8,609,1.377,0.115,1.267,0.195,-0.110,-7.968
9,610,1.354,0.076,1.268,0.176,-0.086,-6.336


In [7]:
# 40-model ensembles
ens = {arm: p.groupby(["siteSR_id", "date", "HUC8"]).agg(y=("y", "first"), pred=("pred", "mean")).reset_index()
       for arm, p in preds.items()}
assert ens["band_only"][["siteSR_id", "date"]].equals(ens["production"][["siteSR_id", "date"]])
pd.DataFrame({arm: M.all_metrics(e["y"], e["pred"]) for arm, e in ens.items()}).T.round(3)

,rmse,mae,mape,bias,p_bias,smape,r2,n
band_only,1.356,1.002,0.434,0.082,2.544,0.336,0.565,1833.0
production,1.257,0.941,0.419,0.152,4.724,0.318,0.627,1833.0


Per-HUC8 holdout RMSE for both ensembles, ordered by sample size. Basins where the gap is large are where the static and weather context carries information that the spectra alone don't.

In [8]:
huc8 = []
for arm, e in ens.items():
    g = e.groupby("HUC8").agg(n=("y", "size"), rmse=("y", lambda y: M.rmse(y, e.loc[y.index, "pred"])))
    huc8.append(g.rename(columns={"rmse": f"{arm}_rmse"}))
huc8 = huc8[0].join(huc8[1].drop(columns="n"))
huc8["delta_rmse"] = huc8["production_rmse"] - huc8["band_only_rmse"]
huc8.sort_values("n", ascending=False).round(3).head(20)

,n,band_only_rmse,production_rmse,delta_rmse
HUC8,,,,
14020002,298,1.017,0.977,-0.040
14010001,225,1.512,1.274,-0.238
14040106,204,1.475,1.401,-0.074
17010303,165,2.062,1.905,-0.157
14080101,117,1.271,1.136,-0.135
10190007,109,1.067,0.943,-0.124
14070003,104,1.260,1.269,0.009
14060007,87,1.123,1.137,0.013
17050116,74,0.582,0.568,-0.015


Feature selection frequency across the 10 band-only seeds.

In [9]:
sel = {}
for seed in SEEDS:
    with open(MODELS_DIR / f"seed{seed}" / "backward_elim_summary.json") as fh:
        for f in json.load(fh)["final_features"]:
            sel[f] = sel.get(f, 0) + 1
pd.Series(sel, name="n_seeds_selected").reindex(BAND_ONLY_CANDIDATES).fillna(0).astype(int).sort_values(ascending=False)

atm_corr_LaSRC    10
temp_corr7        10
NDWI              10
NDSSI             10
fai               10
NR                10
BG                10
BR                10
nir_corr7         10
green_corr7       10
MNDWI             10
swir1_corr7        9
NDVI               9
red_corr7          0
swir2_corr7        0
GR                 0
blue_corr7         0
Name: n_seeds_selected, dtype: int64